In [1]:
import pandas as pd
import numpy as np
import pickle

In [38]:
edges_path = '/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/2019/9606.protein.links.v10.5.txt'
edge_df = pd.read_csv(edges_path, sep=' ')

In [39]:
edge_df.head(2)

,protein1,protein2,combined_score
0,9606.ENSP00000000233,9606.ENSP00000263431,260
1,9606.ENSP00000000233,9606.ENSP00000353863,164


In [41]:
proteins = set(edge_df['protein1'].str.split('.').str[1].tolist())

In [42]:
proteins2 = set(edge_df['protein2'].str.split('.').str[1].tolist())

In [43]:
len(proteins), len(proteins2), len(proteins | proteins2)

(19576, 19576, 19576)

In [44]:
proteins = proteins | proteins2
del proteins2
len(proteins)

19576

In [47]:
import mygene
def get_map_df(ensembl_ids,input_type):
    mg = mygene.MyGeneInfo()
    # Query mygene for UniProt and Entrez gene ID mappings
    results = mg.querymany(
        ensembl_ids,
        scopes=input_type,
        fields='uniprot,entrezgene',
        species='human'
    )

    results_df = pd.DataFrame(results)
    results_df = results_df[~results_df['entrezgene'].isna()]
    results_df['uniprot_ids'] = results_df['uniprot'].apply(
        lambda x: list(x.values())[0] if isinstance(x, dict) and 'Swiss-Prot' in x else None)
    results_df = results_df[~results_df['uniprot_ids'].isna()]
    results_df[results_df['uniprot_ids'].apply(lambda x: isinstance(x, list) and len(x) > 1)]
    return results_df

ppi_ids_map = get_map_df(list(proteins),'ensembl.protein')
ppi_set = set()
for values in ppi_ids_map['uniprot_ids']:
    if isinstance(values, list) and len(values) > 1:
        ppi_set.update(values)  # Add all elements in the list
    else:
        ppi_set.add(values)

string_ids = []
one2more = []
more2one = []  # to collect subdfs with multiple or zero matches

for uniport_ids in list(ppi_set):
    subdf = ppi_ids_map[ppi_ids_map['uniprot_ids'].str.contains(uniport_ids, na=False)]
    
    if len(subdf) == 1:
        if isinstance(subdf['uniprot_ids'], list) and len(values) > 1:
            one2more.append(subdf)
        else:
            string_ids.append(uniport_ids)
    else:
        more2one.append(subdf)

more2one_df = pd.concat(more2one, ignore_index=True)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
2 input query terms found dup hits:	[('ENSP00000319979', 2), ('ENSP00000473163', 3)]
1957 input query terms found no hit:	['ENSP00000358062', 'ENSP00000409932', 'ENSP00000340396', 'ENSP00000447987', 'ENSP00000390651', 'ENS


In [48]:
uni_df = ppi_ids_map[ppi_ids_map['uniprot_ids'].isin(string_ids)][['query','uniprot_ids']]
mapping = dict(zip(uni_df.iloc[:, 0], uni_df.iloc[:, 1]))

In [53]:
more2one_df = more2one_df[['query','uniprot_ids']]
for k, v in zip(more2one_df.iloc[:, 0], more2one_df.iloc[:, 1]):
    mapping[k] = v

In [54]:
len(mapping)

17303

In [56]:
edge_df['p1'] = edge_df['protein1'].str.split('.').str[1]
edge_df['p2'] = edge_df['protein2'].str.split('.').str[1]

In [58]:
edge_df = edge_df[['p1','p2']]

In [61]:
edge_df['p1'] = edge_df['p1'].map(mapping).fillna(edge_df['p1'])
edge_df['p2'] = edge_df['p2'].map(mapping).fillna(edge_df['p2'])

In [64]:
edge_df = edge_df[~edge_df.astype(str).apply(lambda row: row.str.contains('ENSP').any(), axis=1)]


In [66]:
edge_df = edge_df.dropna()

In [67]:
edge_df


,p1,p2
0,P84085,P05129
1,P84085,Q9UQR1
2,P84085,P30041
3,P84085,O60229
4,P84085,P52747
...,...,...
11350628,Q99633,Q7Z7G8
11350629,Q99633,Q9H7D7
11350630,Q99633,Q9H7D0
11350631,Q99633,Q8IZC7


In [68]:
import networkx as nx
G = nx.from_pandas_edgelist(edge_df, source='p1', target='p2')
edges = list(G.edges())


In [70]:
edges_df = pd.DataFrame(G.edges(), columns=['p1', 'p2'])

In [71]:
edges_df

,p1,p2
0,P84085,P05129
1,P84085,Q9UQR1
2,P84085,P30041
3,P84085,O60229
4,P84085,P52747
...,...,...
4632014,P0DP74,P0DP73
4632015,P0DP74,Q8N7N1
4632016,P0DP73,Q8N7N1
4632017,P58511,Q9H0A9


In [ ]:
edge_df.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/edge_2019.csv',index=False)

In [73]:
import pandas as pd
edge_df = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/edge_2019.csv')


In [76]:

pd.concat([edge_df.index, edge_df.index])

TypeError: cannot concatenate object of type '<class 'pandas.core.indexes.range.RangeIndex'>'; only Series and DataFrame objs are valid

In [34]:
pp1_emb = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/data/stringdb/uniport_ppi_2019.csv')

In [35]:
pp1_emb

,string_id,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128
0,P84085,-0.188154,0.192034,0.021007,-0.115466,-0.106862,0.001464,0.079977,0.308825,-0.013989,...,0.163154,0.025956,-0.154160,0.047249,0.013052,-0.121823,0.261028,0.076985,0.027698,0.038653
1,P05129,-0.056868,0.169329,-0.159240,0.109760,-0.025587,0.082301,0.017834,0.020714,-0.020855,...,0.243663,-0.131249,-0.205958,-0.202946,-0.267179,-0.182676,-0.227924,0.124403,-0.201234,-0.194766
2,Q9UQR1,-0.055811,0.039503,0.155353,0.067647,0.073602,0.088512,0.010504,0.194107,0.187011,...,-0.076926,-0.093258,-0.206983,-0.309286,0.032149,0.053479,0.090693,0.144785,0.106935,0.007952
3,P30041,0.055616,0.071914,0.130302,0.231011,-0.232900,-0.332179,-0.048539,-0.151691,-0.101022,...,0.122422,0.295429,0.307127,0.143637,-0.107835,0.022000,-0.087293,0.116431,0.027974,-0.107305
4,O60229,0.114902,-0.038222,-0.257307,0.227826,-0.029925,-0.114639,0.272444,0.259674,-0.243278,...,0.261553,0.011673,-0.377810,-0.331207,-0.174372,0.354861,0.201936,-0.035171,0.220423,0.106151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17217,Q9ULR0,0.156700,-0.140261,-0.233557,-0.054446,-0.008104,-0.152051,0.452839,0.342485,0.157252,...,-0.200293,0.039357,-0.093801,-0.091990,-0.186262,-0.126452,0.087004,-0.004326,0.000820,-0.230722
17218,Q9ULZ0,0.005465,0.132845,0.092608,0.054171,-0.127636,-0.008218,0.239348,0.199174,0.236967,...,0.148058,-0.234042,0.050049,-0.411731,-0.377933,-0.429177,-0.104348,0.377040,0.326177,0.136785
17219,Q9Y2D5,0.154508,0.077624,-0.005013,-0.218087,-0.361138,-0.032404,0.025238,-0.052106,0.318171,...,0.239688,-0.044044,-0.149524,0.220215,-0.214245,-0.180424,-0.237629,0.085903,0.127616,-0.118947
17220,Q9Y6F7,-0.074235,0.249733,0.277919,-0.113700,-0.135856,-0.143085,0.341815,-0.191519,0.291262,...,0.278169,-0.063059,0.215898,-0.054009,-0.441433,-0.274090,-0.046577,-0.203911,0.306785,-0.154667
